In [ ]:
import torch

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Is CUDA available: {torch.cuda.is_available()}")
print(f"Current device: {torch.cuda.current_device()}")
print(f"Device name: {torch.cuda.get_device_name(0)}")

Using device: cpu
Is CUDA available: False


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [ ]:
!pip install torchview

# Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchsummary import summary
import os

# Define Data Directories

In [ ]:
# Define paths to your dataset
train_dir = '/content/drive/MyDrive/processed_images/train_split'  # Replace with your training directory path
val_dir = '/content/drive/MyDrive/processed_images/val_split'  # Replace with your validation directory path
test_dir = '/content/drive/MyDrive/processed_images/test_split'  # Replace with your test directory path

# Define Data Transformations

In [ ]:
from torchvision import transforms

# Define transformations for training, validation, and testing
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),  # Randomly crop and resize to 224x224
    transforms.RandomHorizontalFlip(),  # Randomly flip images horizontally
    transforms.ToTensor(),  # Convert images to PyTorch tensors
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize with ImageNet stats
])

val_transform = transforms.Compose([
    transforms.Resize(256),  # Resize to 256x256
    transforms.CenterCrop(224),  # Center crop to 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = val_transform  # Use the same transformations for testing

# Load Datasets

In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader

# Load datasets using ImageFolder
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(root=val_dir, transform=val_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

# Print class names
class_names = train_dataset.classes
print(f"Class names: {class_names}")

Class names: ['Normal', 'Pneumonia']


# Create DataLoaders

In [ ]:
# Create DataLoaders with multiple workers for faster loading
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# Print dataset sizes
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Testing samples: {len(test_dataset)}")

Training samples: 9609
Validation samples: 1201
Testing samples: 1202


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


*Define parameters for different DenseNet versions*

In [ ]:
model_parameters = {
    'densenet121': [6, 12, 24, 16],
    'densenet169': [6, 12, 32, 32],
    'densenet201': [6, 12, 48, 32],
    'densenet264': [6, 12, 64, 48]
}

# Growth rate and compression factor
k = 32  # Growth rate
compression_factor = 0.5

# Define the DenseNet Model

*Define DenseLayer*



In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels):
        super(DenseLayer, self).__init__()
        self.BN1 = nn.BatchNorm2d(num_features=in_channels)
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=4 * k, kernel_size=1, stride=1, padding=0, bias=False)
        self.BN2 = nn.BatchNorm2d(num_features=4 * k)
        self.conv2 = nn.Conv2d(in_channels=4 * k, out_channels=k, kernel_size=3, stride=1, padding=1, bias=False)
        self.relu = nn.ReLU()

    def forward(self, x):
        xin = x
        x = self.relu(self.BN1(x))
        x = self.conv1(x)
        x = self.relu(self.BN2(x))
        x = self.conv2(x)
        x = torch.cat([xin, x], dim=1)  # Concatenate input and output along channels
        return x

*Define DenseBlock*

In [ ]:
class DenseBlock(nn.Module):
    def __init__(self, num_layers, in_channels):
        super(DenseBlock, self).__init__()
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            self.layers.append(DenseLayer(in_channels + i * k))  # Add k channels per layer

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

*Define TransitionLayer*

In [ ]:
class TransitionLayer(nn.Module):
    def __init__(self, in_channels, compression_factor):
        super(TransitionLayer, self).__init__()
        self.BN = nn.BatchNorm2d(num_features=in_channels)
        self.conv = nn.Conv2d(in_channels=in_channels, out_channels=int(in_channels * compression_factor), kernel_size=1, stride=1, padding=0, bias=False)
        self.avgpool = nn.AvgPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.BN(x)
        x = self.conv(x)
        x = self.avgpool(x)
        return x

**Define DenseNet**

In [ ]:
class DenseNet(nn.Module):
    def __init__(self, densenet_variant, in_channels=3, num_classes=2):  # num_classes=2 for binary classification
        super(DenseNet, self).__init__()
        # Initial convolution and pooling
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False)
        self.BN1 = nn.BatchNorm2d(num_features=64)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

        # DenseBlocks and TransitionLayers
        self.blocks = nn.ModuleList()
        in_channels_block = 64  # Output channels after initial conv

        for i, num_layers in enumerate(densenet_variant):
            # Add DenseBlock
            self.blocks.append(DenseBlock(num_layers, in_channels_block))
            in_channels_block += num_layers * k  # Update input channels for next block

            # Add TransitionLayer (except after the last block)
            if i != len(densenet_variant) - 1:
                self.blocks.append(TransitionLayer(in_channels_block, compression_factor))
                in_channels_block = int(in_channels_block * compression_factor)  # Update input channels after transition

        # Final layers
        self.BN2 = nn.BatchNorm2d(num_features=in_channels_block)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(in_channels_block, num_classes)

    def forward(self, x):
        # Initial convolution and pooling
        x = self.relu(self.BN1(self.conv1(x)))
        x = self.maxpool(x)

        # DenseBlocks and TransitionLayers
        for block in self.blocks:
            x = block(x)

        # Final layers
        x = self.relu(self.BN2(x))
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

*chose and Create the DenseNet Model* *texte en italique*

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Choose DenseNet version
densenet_version = 'densenet201'  # Change to 'densenet121' 'densenet169', 'densenet201', or 'densenet264' as needed

# Create the DenseNet model
model = DenseNet(model_parameters[densenet_version], in_channels=3, num_classes=2).to(device)

# Print model summary
summary(model, (3, 224, 224))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
       BatchNorm2d-5           [-1, 64, 56, 56]             128
              ReLU-6           [-1, 64, 56, 56]               0
            Conv2d-7          [-1, 128, 56, 56]           8,192
       BatchNorm2d-8          [-1, 128, 56, 56]             256
              ReLU-9          [-1, 128, 56, 56]               0
           Conv2d-10           [-1, 32, 56, 56]          36,864
       DenseLayer-11           [-1, 96, 56, 56]               0
      BatchNorm2d-12           [-1, 96, 56, 56]             192
             ReLU-13           [-1, 96, 56, 56]               0
           Conv2d-14          [-1, 128,

# Define Loss Function and Optimizer

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Use CrossEntropyLoss for classification
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Use Adam optimizer

# Define Training and Validation Functions

In [ ]:
def train(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Compute statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100.0 * correct / total
    return train_loss, train_acc

def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Compute statistics
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_loss = running_loss / len(val_loader)
    val_acc = 100.0 * correct / total
    return val_loss, val_acc

# Train the Model

In [ ]:
num_epochs = 20

for epoch in range(num_epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}], "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

Epoch [1/20], Train Loss: 0.6505, Train Acc: 64.17%, Val Loss: 0.6533, Val Acc: 59.37%
Epoch [2/20], Train Loss: 0.6270, Train Acc: 66.08%, Val Loss: 0.6368, Val Acc: 69.78%
Epoch [3/20], Train Loss: 0.6157, Train Acc: 67.04%, Val Loss: 0.5823, Val Acc: 70.52%
Epoch [4/20], Train Loss: 0.6092, Train Acc: 68.12%, Val Loss: 0.5989, Val Acc: 69.36%
Epoch [5/20], Train Loss: 0.5994, Train Acc: 68.66%, Val Loss: 0.5689, Val Acc: 72.44%
Epoch [6/20], Train Loss: 0.5913, Train Acc: 69.12%, Val Loss: 0.5819, Val Acc: 69.53%
Epoch [7/20], Train Loss: 0.5867, Train Acc: 69.63%, Val Loss: 0.5673, Val Acc: 71.69%
Epoch [8/20], Train Loss: 0.5781, Train Acc: 70.92%, Val Loss: 0.5642, Val Acc: 71.52%
Epoch [9/20], Train Loss: 0.5743, Train Acc: 70.81%, Val Loss: 0.5363, Val Acc: 72.19%
Epoch [10/20], Train Loss: 0.5757, Train Acc: 70.54%, Val Loss: 0.5301, Val Acc: 73.86%
Epoch [11/20], Train Loss: 0.5639, Train Acc: 71.46%, Val Loss: 0.5479, Val Acc: 72.19%
Epoch [12/20], Train Loss: 0.5658, Train 

# Test the Model

In [ ]:
def test(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_acc = 100.0 * correct / total
    print(f"Test Accuracy: {test_acc:.2f}%")

# Test the model
test(model, test_loader, device)

Test Accuracy: 71.80%


In [ ]:
model_save_path = '/content/drive/MyDrive/models/densenet201.pth'  # Save to Google Drive
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to /content/drive/MyDrive/models/densenet201.pth


In [ ]:
import pandas as pd

# Function to get predictions
def get_predictions(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            probs = torch.softmax(outputs, dim=1)

            # Store predictions, labels, and probabilities
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return all_preds, all_labels, all_probs

# Get predictions
preds, labels, probs = get_predictions(model, test_loader, device)

# Create a DataFrame
results_df = pd.DataFrame({
    'True_Labels': labels,
    'Predicted_Labels': preds,
    'Probability_Class_0': [p[0] for p in probs],  # Probability for class 0 (e.g., 'normal')
    'Probability_Class_1': [p[1] for p in probs]   # Probability for class 1 (e.g., 'pneumonia')
})

# Save the DataFrame to a CSV file
results_csv_path = '/content/drive/MyDrive/models/test_predictions_densenet201.csv'
results_df.to_csv(results_csv_path, index=False)
print(f"Test predictions saved to {results_csv_path}")

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Test predictions saved to /content/drive/MyDrive/models/test_predictions_densenet201.csv
